## Data Processing and EDA

#### Categories Breakdown

In [1]:
# Breakdown for ALL categories (total review volume) + NON-food/non-drink view
# Requires duckdb: pip install duckdb

import duckdb
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect(database=":memory:")

# -----------------------------
# 1) review counts per business_id
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE review_counts AS
SELECT business_id, COUNT(*)::BIGINT AS review_count
FROM read_json_auto('{REVIEW_PATH}')
GROUP BY business_id;
""")

# -----------------------------
# 2) business table
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE business AS
SELECT business_id, categories
FROM read_json_auto('{BUSINESS_PATH}');
""")

# -----------------------------
# 3) category breakdown (all categories)
# -----------------------------
category_breakdown = con.execute("""
WITH biz AS (
  SELECT
    b.business_id,
    b.categories,
    COALESCE(r.review_count, 0) AS review_count
  FROM business b
  LEFT JOIN review_counts r
  USING (business_id)
),
cats AS (
  SELECT
    trim(unnest(str_split(categories, ','))) AS category,
    review_count
  FROM biz
  WHERE categories IS NOT NULL AND categories <> ''
)
SELECT
  category,
  SUM(review_count) AS review_count,
  COUNT(*) AS business_category_rows
FROM cats
WHERE category IS NOT NULL AND category <> ''
GROUP BY category
ORDER BY review_count DESC;
""").df()

display(category_breakdown)

# -----------------------------
# 4) Updated NON-food / NON-drink filter (based on your category list)
# -----------------------------
df = category_breakdown.copy()
df["review_count"] = pd.to_numeric(df["review_count"], errors="coerce").fillna(0)
df["business_category_rows"] = pd.to_numeric(df["business_category_rows"], errors="coerce").fillna(0).astype(int)

EXCLUDE_EXACT = {
    "Restaurants", "Food", "Nightlife", "Bars",
    "Beer", "Wine & Spirits",
    "Specialty Food", "Grocery",
    "Coffee & Tea", "Cafes", "Desserts",
    "Bakeries", "Caterers",
    "Lounges", "Pubs",
}

FOOD_KEYWORDS = [
    "restaurant", "food", "dining", "diner", "diners", "eatery", "bistro", "cafe", "cafes",
    "kitchen", "grill", "bbq", "barbecue", "barbeque", "buffet", "catering", "caterers",
    "pizza", "burger", "burgers", "sandwich", "sandwiches", "deli", "delis", "bagel", "bagels",
    "taco", "tacos", "burrito", "burritos", "hot dog", "hot dogs",
    "chicken wings", "wings", "chicken shop", "seafood", "steakhouse", "steakhouses",
    "salad", "salads", "noodle", "noodles", "ramen", "sushi", "poke",
    "breakfast", "brunch",
    "coffee", "tea", "boba", "bubble tea", "juice", "smoothie", "smoothies",
    "dessert", "desserts", "bakery", "bakeries", "donut", "donuts",
    "ice cream", "gelato", "frozen yogurt", "candy", "chocolate",
    "vegan", "vegetarian", "gluten-free", "gluten free",
    "grocery", "international grocery", "ethnic grocery", "butcher", "meat shop", "seafood market",
    "farmers market", "market",
    "beer, wine & spirits", "liquor",
]

DRINK_NIGHTLIFE_KEYWORDS = [
    "bar", "bars", "nightlife", "pub", "pubs", "lounge", "lounges",
    "brewery", "breweries", "distillery", "distilleries",
    "wine bar", "wine bars", "cocktail", "cocktails",
    "whiskey", "champagne", "tiki", "hookah", "cigar",
]

CUISINE_KEYWORDS = [
    "american", "mexican", "italian", "chinese", "japanese", "korean", "thai", "vietnamese",
    "indian", "mediterranean", "greek", "french", "spanish", "turkish", "middle eastern",
    "lebanese", "persian", "iranian", "afghan", "moroccan", "ethiopian",
    "brazilian", "peruvian", "argentine", "colombian", "cuban", "caribbean", "hawaiian",
    "filipino", "malaysian", "indonesian", "cambodian", "laotian", "burmese",
    "taiwanese", "hong kong", "dim sum", "szechuan", "cantonese",
    "asian fusion", "tex-mex", "latin american", "cajun", "creole", "southern",
    "tapas",
]

def _word_pat(s: str) -> str:
    esc = re.escape(s).replace(r"\ ", r"[\s\-]+")
    return rf"\b{esc}\b"

food_re    = re.compile("|".join(f"(?:{_word_pat(k)})" for k in FOOD_KEYWORDS), flags=re.IGNORECASE)
drink_re   = re.compile("|".join(f"(?:{_word_pat(k)})" for k in DRINK_NIGHTLIFE_KEYWORDS), flags=re.IGNORECASE)
cuisine_re = re.compile("|".join(f"(?:{_word_pat(k)})" for k in CUISINE_KEYWORDS), flags=re.IGNORECASE)

ALLOWLIST = {"Barbers", "Barber Shops", "Barber"}

def exclude_category(cat: str) -> bool:
    if not isinstance(cat, str) or not cat.strip():
        return False
    c = cat.strip()
    if c in ALLOWLIST:
        return False
    if c in EXCLUDE_EXACT:
        return True
    if food_re.search(c) or drink_re.search(c) or cuisine_re.search(c):
        return True
    return False

df["exclude_food_drink"] = df["category"].apply(exclude_category)

top_non_food = (
    df[~df["exclude_food_drink"]]
    .sort_values("review_count", ascending=False)
    .reset_index(drop=True)
)

print("Top NON-food/non-drink categories by review volume:")
display(top_non_food.head(50)[["category", "review_count", "business_category_rows"]])

,category,review_count,business_category_rows
0,Restaurants,4724471.0,52268
1,Food,1813593.0,27781
2,Nightlife,1539757.0,12281
3,Bars,1455553.0,11065
4,American (Traditional),1011646.0,8139
...,...,...,...
1306,Hepatologists,5.0,1
1307,Ceremonial Clothing,5.0,1
1308,Lahmacun,5.0,1
1309,DUI Schools,5.0,1


Top NON-food/non-drink categories by review volume:


,category,review_count,business_category_rows
0,Event Planning & Services,609553.0,9895
1,Shopping,523254.0,24395
2,Beauty & Spas,370121.0,14292
3,Arts & Entertainment,345059.0,5434
4,Hotels & Travel,329658.0,5857
5,Automotive,239001.0,10773
6,Home Services,238263.0,14356
7,Venues & Event Spaces,214916.0,2480
8,Active Life,206999.0,7687
9,Local Services,206409.0,11198


#### Cities and State Breakdown

In [2]:
import duckdb
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect(database=":memory:")

# --------------------------------------------------
# 1) Review counts per business
# --------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TABLE review_counts AS
SELECT business_id, COUNT(*)::BIGINT AS review_count
FROM read_json_auto('{REVIEW_PATH}')
GROUP BY business_id;
""")

# --------------------------------------------------
# 2) Business table
# --------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TABLE business AS
SELECT business_id, city, state, categories
FROM read_json_auto('{BUSINESS_PATH}');
""")

# --------------------------------------------------
# 3) Explode categories
# --------------------------------------------------
df = con.execute("""
WITH biz AS (
  SELECT
    b.business_id,
    b.city,
    b.state,
    b.categories,
    COALESCE(r.review_count, 0) AS review_count
  FROM business b
  LEFT JOIN review_counts r
  USING (business_id)
)
SELECT
  city,
  state,
  trim(unnest(str_split(categories, ','))) AS category,
  review_count
FROM biz
WHERE categories IS NOT NULL AND categories <> '';
""").df()

# --------------------------------------------------
# 4) EXACT FILTER YOU PROVIDED
# --------------------------------------------------
EXCLUDE_EXACT = {
    "Restaurants", "Food", "Nightlife", "Bars",
    "Beer", "Wine & Spirits",
    "Specialty Food", "Grocery",
    "Coffee & Tea", "Cafes", "Desserts",
    "Bakeries", "Caterers",
    "Lounges", "Pubs",
}

FOOD_KEYWORDS = [
    "restaurant","food","dining","diner","diners","eatery","bistro","cafe","cafes",
    "kitchen","grill","bbq","barbecue","barbeque","buffet","catering","caterers",
    "pizza","burger","burgers","sandwich","sandwiches","deli","delis","bagel","bagels",
    "taco","tacos","burrito","burritos","hot dog","hot dogs",
    "chicken wings","wings","chicken shop","seafood","steakhouse","steakhouses",
    "salad","salads","noodle","noodles","ramen","sushi","poke",
    "breakfast","brunch",
    "coffee","tea","boba","bubble tea","juice","smoothie","smoothies",
    "dessert","desserts","bakery","bakeries","donut","donuts",
    "ice cream","gelato","frozen yogurt","candy","chocolate",
    "vegan","vegetarian","gluten-free","gluten free",
    "grocery","international grocery","ethnic grocery","butcher","meat shop","seafood market",
    "farmers market","market",
    "beer, wine & spirits","liquor",
]

DRINK_NIGHTLIFE_KEYWORDS = [
    "bar","bars","nightlife","pub","pubs","lounge","lounges",
    "brewery","breweries","distillery","distilleries",
    "wine bar","wine bars","cocktail","cocktails",
    "whiskey","champagne","tiki","hookah","cigar",
]

CUISINE_KEYWORDS = [
    "american","mexican","italian","chinese","japanese","korean","thai","vietnamese",
    "indian","mediterranean","greek","french","spanish","turkish","middle eastern",
    "lebanese","persian","iranian","afghan","moroccan","ethiopian",
    "brazilian","peruvian","argentine","colombian","cuban","caribbean","hawaiian",
    "filipino","malaysian","indonesian","cambodian","laotian","burmese",
    "taiwanese","hong kong","dim sum","szechuan","cantonese",
    "asian fusion","tex-mex","latin american","cajun","creole","southern",
    "tapas",
]

def word_pat(s):
    return r"\b" + re.escape(s).replace(r"\ ", r"[\s\-]+") + r"\b"

food_re    = re.compile("|".join(word_pat(k) for k in FOOD_KEYWORDS), flags=re.I)
drink_re   = re.compile("|".join(word_pat(k) for k in DRINK_NIGHTLIFE_KEYWORDS), flags=re.I)
cuisine_re = re.compile("|".join(word_pat(k) for k in CUISINE_KEYWORDS), flags=re.I)

ALLOWLIST = {"Barbers", "Barber Shops", "Barber"}

def is_non_food(cat):
    if cat in ALLOWLIST:
        return True
    if cat in EXCLUDE_EXACT:
        return False
    if food_re.search(cat) or drink_re.search(cat) or cuisine_re.search(cat):
        return False
    return True

df["is_non_food"] = df["category"].apply(is_non_food)

non_food = df[df["is_non_food"]]

# --------------------------------------------------
# 5) Top 15 cities (NON-food only)
# --------------------------------------------------
top_15_cities = (
    non_food
    .groupby("city", as_index=False)["review_count"]
    .sum()
    .sort_values("review_count", ascending=False)
    .head(15)
)

print("Top 15 cities (NON-food / NON-drink):")
display(top_15_cities)

# --------------------------------------------------
# 6) Top 15 states (NON-food only)
# --------------------------------------------------
top_15_states = (
    non_food
    .groupby("state", as_index=False)["review_count"]
    .sum()
    .sort_values("review_count", ascending=False)
    .head(15)
)

print("Top 15 states (NON-food / NON-drink):")
display(top_15_states)

# --------------------------------------------------
# 5) Top 15 cities (NON-food only) + force PHX/Tempe/Tucson
# --------------------------------------------------

FORCE_CITIES = {
    "Phoenix",
    "Tempe",
    "Tucson"
}

# Normal top 15
top_15_cities = (
    non_food
    .groupby("city", as_index=False)["review_count"]
    .sum()
    .sort_values("review_count", ascending=False)
    .head(15)
)

# Forced cities (even if not top 15)
forced_cities = (
    non_food[non_food["city"].isin(FORCE_CITIES)]
    .groupby("city", as_index=False)["review_count"]
    .sum()
)

Top 15 cities (NON-food / NON-drink):


,city,review_count
760,Philadelphia,1493500
661,New Orleans,802517
1009,Tampa,789758
809,Reno,788143
1055,Tucson,723431
649,Nashville,689845
878,Santa Barbara,535725
441,Indianapolis,517036
862,Saint Louis,385730
77,Boise,172007


Top 15 states (NON-food / NON-drink):


,state,review_count
16,PA,2554432
5,FL,1828489
10,LA,998275
15,NV,964452
18,TN,943484
1,AZ,767344
13,MO,764821
9,IN,712902
2,CA,663378
14,NJ,404197


#### Philly Food Breakdown

In [3]:
import duckdb
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect(database=":memory:")

# --------------------------------------------------
# 1) Review counts per business
# --------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TABLE review_counts AS
SELECT business_id, COUNT(*)::BIGINT AS review_count
FROM read_json_auto('{REVIEW_PATH}')
GROUP BY business_id;
""")

# --------------------------------------------------
# 2) Business table (Philadelphia only)
# --------------------------------------------------
biz = con.execute("""
SELECT
  b.business_id,
  b.city,
  b.state,
  b.categories,
  COALESCE(r.review_count, 0) AS review_count
FROM read_json_auto(? ) b
LEFT JOIN review_counts r
USING (business_id)
WHERE b.city = 'Philadelphia'
  AND b.categories IS NOT NULL
  AND b.categories <> '';
""", [BUSINESS_PATH]).df()

# --------------------------------------------------
# 3) EXACT food / drink filter (same as before)
# --------------------------------------------------
EXCLUDE_EXACT = {
    "Restaurants", "Food", "Nightlife", "Bars",
    "Beer", "Wine & Spirits",
    "Specialty Food", "Grocery",
    "Coffee & Tea", "Cafes", "Desserts",
    "Bakeries", "Caterers",
    "Lounges", "Pubs",
}

FOOD_KEYWORDS = [
    "restaurant","food","dining","diner","diners","eatery","bistro","cafe","cafes",
    "kitchen","grill","bbq","barbecue","barbeque","buffet","catering","caterers",
    "pizza","burger","burgers","sandwich","sandwiches","deli","delis","bagel","bagels",
    "taco","tacos","burrito","burritos","hot dog","hot dogs",
    "chicken wings","wings","chicken shop","seafood","steakhouse","steakhouses",
    "salad","salads","noodle","noodles","ramen","sushi","poke",
    "breakfast","brunch",
    "coffee","tea","boba","bubble tea","juice","smoothie","smoothies",
    "dessert","desserts","bakery","bakeries","donut","donuts",
    "ice cream","gelato","frozen yogurt","candy","chocolate",
    "vegan","vegetarian","gluten-free","gluten free",
    "grocery","international grocery","ethnic grocery","butcher","meat shop","seafood market",
    "farmers market","market",
    "beer, wine & spirits","liquor",
]

DRINK_NIGHTLIFE_KEYWORDS = [
    "bar","bars","nightlife","pub","pubs","lounge","lounges",
    "brewery","breweries","distillery","distilleries",
    "wine bar","wine bars","cocktail","cocktails",
    "whiskey","champagne","tiki","hookah","cigar",
]

CUISINE_KEYWORDS = [
    "american","mexican","italian","chinese","japanese","korean","thai","vietnamese",
    "indian","mediterranean","greek","french","spanish","turkish","middle eastern",
    "lebanese","persian","iranian","afghan","moroccan","ethiopian",
    "brazilian","peruvian","argentine","colombian","cuban","caribbean","hawaiian",
    "filipino","malaysian","indonesian","cambodian","laotian","burmese",
    "taiwanese","hong kong","dim sum","szechuan","cantonese",
    "asian fusion","tex-mex","latin american","cajun","creole","southern",
    "tapas",
]

def word_pat(s):
    return r"\b" + re.escape(s).replace(r"\ ", r"[\s\-]+") + r"\b"

food_re    = re.compile("|".join(word_pat(k) for k in FOOD_KEYWORDS), flags=re.I)
drink_re   = re.compile("|".join(word_pat(k) for k in DRINK_NIGHTLIFE_KEYWORDS), flags=re.I)
cuisine_re = re.compile("|".join(word_pat(k) for k in CUISINE_KEYWORDS), flags=re.I)

def is_food_or_drink(cat):
    if cat in EXCLUDE_EXACT:
        return True
    if food_re.search(cat) or drink_re.search(cat) or cuisine_re.search(cat):
        return True
    return False

# --------------------------------------------------
# 4) Business-level food flag
# --------------------------------------------------
biz["category_list"] = biz["categories"].apply(lambda s: [c.strip() for c in s.split(",")])
biz["is_food_business"] = biz["category_list"].apply(
    lambda cats: any(is_food_or_drink(c) for c in cats)
)

food_biz = biz[biz["is_food_business"]]

# --------------------------------------------------
# 5) FINAL ANSWER
# --------------------------------------------------
total_food_reviews_philly = food_biz["review_count"].sum()
num_food_businesses = food_biz.shape[0]

print("Philadelphia — FOOD / DRINK businesses only")
print(f"Total reviews: {total_food_reviews_philly:,}")
print(f"Number of food/drink businesses: {num_food_businesses:,}")

Philadelphia — FOOD / DRINK businesses only
Total reviews: 764,854
Number of food/drink businesses: 7,608


In [4]:
import duckdb
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect(database=":memory:")

# --------------------------------------------------
# 1) Review counts per business
# --------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TABLE review_counts AS
SELECT business_id, COUNT(*)::BIGINT AS review_count
FROM read_json_auto('{REVIEW_PATH}')
GROUP BY business_id;
""")

# --------------------------------------------------
# 2) Philadelphia businesses only
# --------------------------------------------------
df = con.execute("""
SELECT
  b.business_id,
  b.categories,
  COALESCE(r.review_count, 0) AS review_count
FROM read_json_auto(?) b
LEFT JOIN review_counts r
USING (business_id)
WHERE b.city = 'Philadelphia'
  AND b.categories IS NOT NULL
  AND b.categories <> '';
""", [BUSINESS_PATH]).df()

# --------------------------------------------------
# 3) EXACT FILTER (your version)
# --------------------------------------------------
EXCLUDE_EXACT = {
    "Restaurants", "Food", "Nightlife", "Bars",
    "Beer", "Wine & Spirits",
    "Specialty Food", "Grocery",
    "Coffee & Tea", "Cafes", "Desserts",
    "Bakeries", "Caterers",
    "Lounges", "Pubs",
}

FOOD_KEYWORDS = [
    "restaurant","food","dining","diner","diners","eatery","bistro","cafe","cafes",
    "kitchen","grill","bbq","barbecue","barbeque","buffet","catering","caterers",
    "pizza","burger","burgers","sandwich","sandwiches","deli","delis","bagel","bagels",
    "taco","tacos","burrito","burritos","hot dog","hot dogs",
    "chicken wings","wings","chicken shop","seafood","steakhouse","steakhouses",
    "salad","salads","noodle","noodles","ramen","sushi","poke",
    "breakfast","brunch",
    "coffee","tea","boba","bubble tea","juice","smoothie","smoothies",
    "dessert","desserts","bakery","bakeries","donut","donuts",
    "ice cream","gelato","frozen yogurt","candy","chocolate",
    "vegan","vegetarian","gluten-free","gluten free",
    "grocery","international grocery","ethnic grocery","butcher","meat shop","seafood market",
    "farmers market","market",
    "beer, wine & spirits","liquor",
]

DRINK_NIGHTLIFE_KEYWORDS = [
    "bar","bars","nightlife","pub","pubs","lounge","lounges",
    "brewery","breweries","distillery","distilleries",
    "wine bar","wine bars","cocktail","cocktails",
    "whiskey","champagne","tiki","hookah","cigar",
]

CUISINE_KEYWORDS = [
    "american","mexican","italian","chinese","japanese","korean","thai","vietnamese",
    "indian","mediterranean","greek","french","spanish","turkish","middle eastern",
    "lebanese","persian","iranian","afghan","moroccan","ethiopian",
    "brazilian","peruvian","argentine","colombian","cuban","caribbean","hawaiian",
    "filipino","malaysian","indonesian","cambodian","laotian","burmese",
    "taiwanese","hong kong","dim sum","szechuan","cantonese",
    "asian fusion","tex-mex","latin american","cajun","creole","southern",
    "tapas",
]

def word_pat(s):
    return r"\b" + re.escape(s).replace(r"\ ", r"[\s\-]+") + r"\b"

food_re    = re.compile("|".join(word_pat(k) for k in FOOD_KEYWORDS), flags=re.I)
drink_re   = re.compile("|".join(word_pat(k) for k in DRINK_NIGHTLIFE_KEYWORDS), flags=re.I)
cuisine_re = re.compile("|".join(word_pat(k) for k in CUISINE_KEYWORDS), flags=re.I)

def is_food_category(cat):
    if cat in EXCLUDE_EXACT:
        return True
    if food_re.search(cat) or drink_re.search(cat) or cuisine_re.search(cat):
        return True
    return False

# --------------------------------------------------
# 4) Explode categories + keep FOOD ONLY
# --------------------------------------------------
df["category_list"] = df["categories"].apply(
    lambda s: [c.strip() for c in s.split(",") if c.strip()]
)

exploded = df.explode("category_list").rename(columns={"category_list": "category"})

food_only = exploded[exploded["category"].apply(is_food_category)]

# --------------------------------------------------
# 5) Food-type distribution
# --------------------------------------------------
philly_food_dist = (
    food_only
    .groupby("category", as_index=False)["review_count"]
    .sum()
    .sort_values("review_count", ascending=False)
)

print("Philadelphia — FOOD / DRINK category distribution:")
display(philly_food_dist.head(30))

Philadelphia — FOOD / DRINK category distribution:


,category,review_count
110,Restaurants,687289
55,Food,271182
100,Nightlife,235223
10,Bars,223180
2,American (New),153715
16,Breakfast & Brunch,129387
3,American (Traditional),114673
112,Sandwiches,112915
35,Coffee & Tea,84760
81,Italian,82982


### Hotels and Music Venue EDA

#### Number of Reviews

In [5]:
import duckdb
from pathlib import Path
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

# -----------------------------
# DuckDB in-memory connection
# -----------------------------
con = duckdb.connect(database=":memory:")

# -----------------------------
# 1) Review counts per business_id
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE review_counts AS
SELECT business_id, COUNT(*)::BIGINT AS review_count
FROM read_json_auto('{REVIEW_PATH}')
GROUP BY business_id;
""")

# -----------------------------
# 2) Business table (only what we need)
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE business AS
SELECT business_id, categories
FROM read_json_auto('{BUSINESS_PATH}');
""")

# -----------------------------
# 3) Total review volume for Hotels + Music Venues
# -----------------------------
target_reviews = con.execute("""
WITH biz AS (
  SELECT
    b.business_id,
    b.categories,
    COALESCE(r.review_count, 0) AS review_count
  FROM business b
  LEFT JOIN review_counts r
  USING (business_id)
),
cats AS (
  SELECT
    business_id,
    trim(unnest(str_split(categories, ','))) AS category,
    review_count
  FROM biz
  WHERE categories IS NOT NULL AND categories <> ''
)
SELECT
  category,
  SUM(review_count) AS total_reviews,
  COUNT(DISTINCT business_id) AS num_businesses
FROM cats
WHERE category IN ('Hotels', 'Music Venues')
GROUP BY category
ORDER BY total_reviews DESC;
""").df()

print("Total review volume for Hotels and Music Venues:")
display(target_reviews)

Total review volume for Hotels and Music Venues:


,category,total_reviews,num_businesses
0,Hotels,190269.0,2977
1,Music Venues,118777.0,1104


#### Location Distribution

In [6]:
import duckdb
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect(database=":memory:")

# Business (need city + categories)
con.execute(f"""
CREATE OR REPLACE TABLE business AS
SELECT business_id, city, categories
FROM read_json_auto('{BUSINESS_PATH}');
""")

# Reviews (business_id only)
con.execute(f"""
CREATE OR REPLACE TABLE reviews AS
SELECT business_id
FROM read_json_auto('{REVIEW_PATH}');
""")

# Top 10 cities per category
top_10_cities_by_category = con.execute("""
WITH cats AS (
  SELECT
    b.business_id,
    b.city,
    trim(unnest(str_split(b.categories, ','))) AS category
  FROM business b
  WHERE b.categories IS NOT NULL AND b.categories <> ''
),
filtered AS (
  SELECT
    c.category,
    c.city
  FROM cats c
  JOIN reviews r
    ON c.business_id = r.business_id
  WHERE c.category IN ('Hotels', 'Music Venues')
),
ranked AS (
  SELECT
    category,
    city,
    COUNT(*) AS total_reviews,
    ROW_NUMBER() OVER (
      PARTITION BY category
      ORDER BY COUNT(*) DESC
    ) AS rn
  FROM filtered
  GROUP BY category, city
)
SELECT
  category,
  city,
  total_reviews
FROM ranked
WHERE rn <= 10
ORDER BY category, total_reviews DESC;
""").df()

print("Top 10 cities by total review volume (separate for Hotels vs Music Venues):")
display(top_10_cities_by_category)

Top 10 cities by total review volume (separate for Hotels vs Music Venues):


,category,city,total_reviews
0,Hotels,New Orleans,33241
1,Hotels,Reno,18231
2,Hotels,Santa Barbara,16336
3,Hotels,Nashville,16269
4,Hotels,Philadelphia,14378
5,Hotels,Tucson,10411
6,Hotels,Tampa,9814
7,Hotels,Indianapolis,7665
8,Hotels,Saint Louis,5451
9,Hotels,St. Pete Beach,3408


#### Year Distribution for Music Venues

In [7]:
import duckdb
from pathlib import Path
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

# -----------------------------
# DuckDB connection
# -----------------------------
con = duckdb.connect(database=":memory:")

# -----------------------------
# 1) Business table
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE business AS
SELECT business_id, city, categories
FROM read_json_auto('{BUSINESS_PATH}');
""")

# -----------------------------
# 2) Reviews table (need date)
# -----------------------------
con.execute(f"""
CREATE OR REPLACE TABLE reviews AS
SELECT
  business_id,
  CAST(date AS DATE) AS review_date
FROM read_json_auto('{REVIEW_PATH}');
""")

# -----------------------------
# 3) Year-by-year distribution — Music Venues in Nashville
# -----------------------------
year_distribution_nashville = con.execute("""
WITH cats AS (
  SELECT
    b.business_id,
    b.city,
    trim(unnest(str_split(b.categories, ','))) AS category
  FROM business b
  WHERE b.categories IS NOT NULL AND b.categories <> ''
),
filtered AS (
  SELECT
    EXTRACT(YEAR FROM r.review_date) AS year
  FROM cats c
  JOIN reviews r
    ON c.business_id = r.business_id
  WHERE c.category = 'Music Venues'
    AND LOWER(c.city) = 'nashville'
)
SELECT
  year,
  COUNT(*) AS num_reviews
FROM filtered
GROUP BY year
ORDER BY year;
""").df()

print("Year-by-year review distribution — Music Venues in Nashville:")
display(year_distribution_nashville)

# -----------------------------
# 3) Year-by-year distribution — Music Venues in Nashville
#    + number of businesses
# -----------------------------
year_distribution_nashville = con.execute("""
WITH cats AS (
  SELECT
    b.business_id,
    b.city,
    trim(unnest(str_split(b.categories, ','))) AS category
  FROM business b
  WHERE b.categories IS NOT NULL AND b.categories <> ''
),
filtered AS (
  SELECT
    c.business_id,
    EXTRACT(YEAR FROM r.review_date) AS year
  FROM cats c
  JOIN reviews r
    ON c.business_id = r.business_id
  WHERE c.category = 'Music Venues'
    AND LOWER(c.city) = 'nashville'
)
SELECT
  year,
  COUNT(*) AS num_reviews,
  COUNT(DISTINCT business_id) AS num_businesses
FROM filtered
GROUP BY year
ORDER BY year;
""").df()

print("Year-by-year review distribution — Music Venues in Nashville:")
display(year_distribution_nashville)

Year-by-year review distribution — Music Venues in Nashville:


,year,num_reviews
0,2005,5
1,2006,19
2,2007,107
3,2008,398
4,2009,280
5,2010,496
6,2011,938
7,2012,1089
8,2013,1583
9,2014,2137


Year-by-year review distribution — Music Venues in Nashville:


,year,num_reviews,num_businesses
0,2005,5,5
1,2006,19,12
2,2007,107,33
3,2008,398,53
4,2009,280,55
5,2010,496,65
6,2011,938,77
7,2012,1089,83
8,2013,1583,94
9,2014,2137,98


#### Total Number of Reviews

In [1]:
import duckdb
from pathlib import Path

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect()

query = f"""
WITH music_venues_nashville AS (
    SELECT business_id
    FROM read_json_auto('{BUSINESS_PATH}')
    WHERE city = 'Nashville'
      AND state = 'TN'
      AND categories ILIKE '%Music Venues%'
),
reviews_filtered AS (
    SELECT 
        r.review_id,
        r.business_id,
        EXTRACT(YEAR FROM r.date) AS year
    FROM read_json_auto('{REVIEW_PATH}') r
    JOIN music_venues_nashville m
      ON r.business_id = m.business_id
    WHERE EXTRACT(YEAR FROM r.date) BETWEEN 2013 AND 2019
)
SELECT 
    year,
    COUNT(*) AS total_reviews
FROM reviews_filtered
GROUP BY year
ORDER BY year;
"""

df = con.execute(query).fetchdf()

print(df)

# If you also want the grand total across 2013–2019:
grand_total = df["total_reviews"].sum()
print("\nGrand Total (2013–2019):", grand_total)

   year  total_reviews
0  2013           1583
1  2014           2137
2  2015           2535
3  2016           2548
4  2017           2661
5  2018           3794
6  2019           3667

Grand Total (2013–2019): 18925


#### Month Breakdown

In [2]:
import duckdb
from pathlib import Path

DATA_DIR = Path("../yelp_dataset")
BUSINESS_PATH = str(DATA_DIR / "yelp_academic_dataset_business.json")
REVIEW_PATH   = str(DATA_DIR / "yelp_academic_dataset_review.json")

con = duckdb.connect()

query = f"""
WITH music_venues_nashville AS (
    SELECT business_id
    FROM read_json_auto('{BUSINESS_PATH}')
    WHERE city = 'Nashville'
      AND state = 'TN'
      AND categories ILIKE '%Music Venues%'
),
reviews_filtered AS (
    SELECT 
        EXTRACT(MONTH FROM r.date) AS month
    FROM read_json_auto('{REVIEW_PATH}') r
    JOIN music_venues_nashville m
      ON r.business_id = m.business_id
    WHERE EXTRACT(YEAR FROM r.date) BETWEEN 2013 AND 2019
)
SELECT 
    month,
    COUNT(*) AS total_reviews
FROM reviews_filtered
GROUP BY month
ORDER BY total_reviews DESC;
"""

df = con.execute(query).fetchdf()

print(df)

    month  total_reviews
0       7           1931
1       6           1837
2       5           1766
3       8           1762
4      10           1743
5       3           1703
6       9           1583
7       4           1546
8      11           1361
9       1           1300
10     12           1232
11      2           1161
